# Notebook 13 — Thử nghiệm kiến trúc DL mới

Mỗi cell thí nghiệm tương ứng một ý tưởng theo thứ tự ưu tiên đã thống nhất. Hiện có ba cell chạy độc lập: FiLM, low-rank fusion và Hadamard fusion. Cell cuối để tổng hợp và so sánh với hai file global cũ sẽ được thêm sau khi hoàn tất các ý tưởng.


In [7]:
from datetime import datetime
from pathlib import Path
import os
import sys
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'controllers').is_dir() and (p / 'experiments').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repository root from {cwd}')
sys.path.insert(0, str(REPO_ROOT))

from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline

BASE_DIR = 'D:/variant_data'
FM_PROFILE_JSON = f'{BASE_DIR}/profiling/fm_profiling_sequence_ablation.json'
FM_SUPPORT_JSON = f'{BASE_DIR}/profiling/fm_sequence_ablation_support.json'
EMBEDDING_DIR = f'{BASE_DIR}/fm_embeddings/sequence_ablation'
GEOMETRY_K = 64
GEOMETRY_DIR = f'{BASE_DIR}/geometry_v2_k{GEOMETRY_K}/{{context}}'
GEOM_PROFILE_JSON = None
GEOM_FEATURE_NAMES = [f'{m}_{f}' for m in ('dna', 'prot') for f in ('LVD_L2', 'LVD_Cosine', 'LID', 'LVD_Relative')]
TARGET_CONFIGS = [
    {'name': 'A_no_geometry', 'ablation': 'bio_core_dna_prot_spliceai', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai'], 'geom_feature_names': None},
    {'name': 'B_with_geometry', 'ablation': 'bio_core_dna_prot_spliceai_geom', 'active_modalities': ['dna', 'prot', 'bio_core', 'spliceai', 'geom'], 'geom_feature_names': GEOM_FEATURE_NAMES},
]
DNA_MODEL = {'name': 'nt_v1_500m_dna_1001', 'seq_type': 'dna', 'sequence_length': 1001}
PROTEIN_MODEL = {'name': 'esm2_650m_prot_501', 'seq_type': 'protein', 'sequence_length': 501}
MODELS_SPACE = [DNA_MODEL, PROTEIN_MODEL]
DATASETS = [
    {'name': 'Train3Val_Test', 'train': 'train3', 'val': 'val', 'test': 'test'},
    {'name': 'Train3Val_ClinVarHQ', 'train': 'train3', 'val': 'val', 'test': 'clinvarhq'},
    {'name': 'Train3Val_UniProt', 'train': 'train3', 'val': 'val', 'test': 'uniprot'},
    {'name': 'Train3Val_ProteinGym', 'train': 'train3', 'val': 'val', 'test': 'proteingym'},
]
CONFIG = {'batch_size': 256, 'epochs': 30, 'lr': 1e-4, 'weight_decay': 1e-3, 'lr_factor': 0.5, 'lr_patience': 3, 'early_stop_patience': 6, 'num_workers': 0, 'seed': 42, 'xgb_n_jobs': 1}
EXPERIMENTS = [{'name': 'PyTorch_Concat', 'type': 'pytorch', 'fusion': 'concat'}]
EXPLAINABILITY = {'enable_shap': False, 'enable_lime': False}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

RESUME_EXISTING_RUN = False
RESUME_BATCH_RUN_DIR = None
if RESUME_EXISTING_RUN:
    if RESUME_BATCH_RUN_DIR is None:
        raise ValueError('Set RESUME_BATCH_RUN_DIR when RESUME_EXISTING_RUN=True')
    RUN_OUTPUT_DIR = Path(RESUME_BATCH_RUN_DIR).resolve()
else:
    RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
    RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}'
    suffix = 2
    while RUN_OUTPUT_DIR.exists():
        RUN_OUTPUT_DIR = REPO_ROOT / 'experiments' / f'module06_architecture_run_{RUN_STAMP}_{suffix:02d}'
        suffix += 1

pipeline = FusionBatchPipeline(
    base_dir=BASE_DIR, config=CONFIG, datasets=DATASETS, models_space=MODELS_SPACE,
    pooling_strategies=['center'], experiments=EXPERIMENTS, explainability=EXPLAINABILITY,
    fm_profile_json=FM_PROFILE_JSON, geom_profile_json=GEOM_PROFILE_JSON,
    geometry_dir=GEOMETRY_DIR, geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(RUN_OUTPUT_DIR), embedding_dir=EMBEDDING_DIR,
)
print(f'Device: {DEVICE}')
print(f'Batch run directory: {pipeline.batch_run_dir}')
print(f'Target configurations: {[c["name"] for c in TARGET_CONFIGS]}')

Device: cuda
Batch run directory: D:\Missense_variant_pathogenicity_prediction\experiments\module06_architecture_run_20261102_022601
Target configurations: ['A_no_geometry', 'B_with_geometry']


## Ưu tiên đầu — FiLM có bio/geometry làm ngữ cảnh

Bio/geometry sinh tham số FiLM để điều chế nhánh DNA và protein. Grid 16 tổ hợp: hidden_dim {512, 256}, dropout {0.2, 0.1}, learning_rate {1e-4, 5e-5}, weight_decay {1e-3, 1e-4}; trial đầu là cấu hình FiLM baseline. Mỗi trial chạy A/B, chọn checkpoint theo validation MCC và đánh giá bốn tập test.

Mỗi trial có checkpoint và manifest riêng để resume; cell lưu thêm các CSV grid tổng hợp dưới thư mục FiLM.

In [5]:
import importlib
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
film_experiment_module = importlib.import_module("core.module06_architecture_experiments.film_experiment")
film_experiment_module = importlib.reload(film_experiment_module)
get_default_film_config = film_experiment_module.get_default_film_config
get_default_film_grid = film_experiment_module.get_default_film_grid
run_film_grid_experiment = film_experiment_module.run_film_grid_experiment

FILM_CONFIG = get_default_film_config()
FILM_GRID_SEARCH = get_default_film_grid()
# Edit the value lists here. This is a 2x2x2x2 grid (16 trials).
FILM_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
FILM_RESULTS = run_film_grid_experiment(
    FILM_CONTEXT, base_config=FILM_CONFIG, search_space=FILM_GRID_SEARCH, display_fn=display
)


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.2971 val_loss=0.1883 val_MCC=0.8067 threshold=0.7554
[FiLM] A_no_geometry epoch 2/30 train_loss=0.2231 val_loss=0.1884 val_MCC=0.8133 threshold=0.7663
[FiLM] A_no_geometry epoch 3/30 train_loss=0.1947 val_loss=0.1723 val_MCC=0.8138 threshold=0.7226
[FiLM] A_no_geometry epoch 4/30 train_loss=0.1617 val_loss=0.1914 val_MCC=0.8072 threshold=0.8441
[FiLM] A_no_geometry epoch 5/30 train_loss=0.1317 val_loss=0.2205 val_MCC=0.7975 threshold=0.9244
[FiLM] A

"[FiLM grid] trial_001/16 complete: {'hidden_dim': 512, 'dropout': 0.2, 'learning_rate': 0.0001, 'weight_decay': 0.001}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.2971 val_loss=0.1883 val_MCC=0.8067 threshold=0.7554
[FiLM] A_no_geometry epoch 2/30 train_loss=0.2231 val_loss=0.1884 val_MCC=0.8133 threshold=0.7663
[FiLM] A_no_geometry epoch 3/30 train_loss=0.1947 val_loss=0.1723 val_MCC=0.8138 threshold=0.7226
[FiLM] A_no_geometry epoch 4/30 train_loss=0.1617 val_loss=0.1914 val_MCC=0.8072 threshold=0.8441
[FiLM] A_no_geometry epoch 5/30 train_loss=0.1317 val_loss=0.2205 val_MCC=0.7975 threshold=0.9244
[FiLM] A

"[FiLM grid] trial_002/16 complete: {'hidden_dim': 512, 'dropout': 0.2, 'learning_rate': 0.0001, 'weight_decay': 0.0001}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[FiLM] A_no_geometry epoch 1/30 train_loss=0.3326 val_loss=0.2017 val_MCC=0.8041 threshold=0.7760
[FiLM] A_no_geometry epoch 2/30 train_loss=0.2378 val_loss=0.1770 val_MCC=0.8084 threshold=0.7535
[FiLM] A_no_geometry epoch 3/30 train_loss=0.2178 val_loss=0.1852 val_MCC=0.8136 threshold=0.7433
[FiLM] A_no_geometry epoch 4/30 train_loss=0.1960 val_loss=0.1765 val_MCC=0.8125 threshold=0.7244
[FiLM] A_no_geometry epoch 5/30 train_loss=0.1798 val_loss=0.1821 val_MCC=0.8118 threshold=0.7250
[FiLM] A

"[FiLM grid] trial_003/16 complete: {'hidden_dim': 512, 'dropout': 0.2, 'learning_rate': 5e-05, 'weight_decay': 0.001}"

... (25 outputs left out)

"[FiLM grid] trial_016/16 complete: {'hidden_dim': 256, 'dropout': 0.1, 'learning_rate': 5e-05, 'weight_decay': 0.0001}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Source_File,Val_Accuracy,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Grid_Hyperparameters
0,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9429,0.8575,0.8405,0.9671,0.8489,0.9752,0.9159,trial_001,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
2,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9429,0.8575,0.8405,0.9671,0.8489,0.9752,0.9159,trial_002,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
12,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9432,0.8649,0.8324,0.9693,0.8483,0.9750,0.9139,trial_007,"{""dropout"": 0.1, ""hidden_dim"": 512, ""learning_..."
14,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9432,0.8649,0.8324,0.9693,0.8483,0.9750,0.9139,trial_008,"{""dropout"": 0.1, ""hidden_dim"": 512, ""learning_..."
4,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9423,0.8459,0.8527,0.9634,0.8493,0.9749,0.9142,trial_003,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
6,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9423,0.8459,0.8527,0.9634,0.8493,0.9749,0.9142,trial_004,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
20,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9428,0.8597,0.8366,0.9678,0.8480,0.9750,0.9165,trial_011,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
22,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9428,0.8597,0.8366,0.9678,0.8480,0.9750,0.9165,trial_012,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
24,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9426,0.8596,0.8359,0.9678,0.8476,0.9745,0.9147,trial_013,"{""dropout"": 0.1, ""hidden_dim"": 256, ""learning_..."
26,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,FiLM,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9426,0.8596,0.8359,0.9678,0.8476,0.9745,0.9147,trial_014,"{""dropout"": 0.1, ""hidden_dim"": 256, ""learning_..."


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,AUPRC,Trial_ID,Grid_Hyperparameters
65,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.9388,0.9635,0.9109,0.9662,0.9365,0.8789,0.9849,0.9844,trial_009,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
73,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.9388,0.9635,0.9109,0.9662,0.9365,0.8789,0.9849,0.9844,trial_010,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
81,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.9374,0.9691,0.9023,0.9718,0.9345,0.8768,0.9846,0.9842,trial_011,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
89,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.9374,0.9691,0.9023,0.9718,0.9345,0.8768,0.9846,0.9842,trial_012,"{""dropout"": 0.2, ""hidden_dim"": 256, ""learning_..."
113,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.9360,0.9550,0.9138,0.9577,0.9339,0.8727,0.9842,0.9839,trial_015,"{""dropout"": 0.1, ""hidden_dim"": 256, ""learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.7284,0.7254,0.7319,0.7250,0.7286,0.4569,0.7928,0.7721,trial_016,"{""dropout"": 0.1, ""hidden_dim"": 256, ""learning_..."
6,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.7277,0.7277,0.7244,0.7309,0.7260,0.4554,0.7949,0.7815,trial_001,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
14,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.7277,0.7277,0.7244,0.7309,0.7260,0.4554,0.7949,0.7815,trial_002,"{""dropout"": 0.2, ""hidden_dim"": 512, ""learning_..."
38,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,FiLM,...,0.7202,0.7058,0.7515,0.6891,0.7279,0.4414,0.7907,0.7781,trial_005,"{""dropout"": 0.1, ""hidden_dim"": 512, ""learning_..."


'FiLM grid complete (16 trials). Aggregate CSVs: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261102_012820\\module06_architecture_experiments\\film'

## Ưu tiên đầu — Low-rank multimodal fusion

Chiếu riêng DNA delta, protein delta và bio/geometry. Tích Hadamard ba factor hạng thấp rồi cộng theo rank tạo tương tác; skip path giữ tín hiệu từng nhánh. Grid 16 tổ hợp: rank {8, 4}, fusion_dim {256, 128}, dropout {0.2, 0.1}, learning_rate {1e-4, 5e-5}; trial đầu là cấu hình low-rank baseline. Mỗi trial chạy A/B, chọn checkpoint theo validation MCC và đánh giá bốn tập test.

Mỗi trial có checkpoint và manifest riêng để resume; cell lưu thêm các CSV grid tổng hợp dưới thư mục low_rank.

In [6]:
import importlib
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
low_rank_experiment_module = importlib.import_module("core.module06_architecture_experiments.low_rank_experiment")
low_rank_experiment_module = importlib.reload(low_rank_experiment_module)
LOW_RANK_CONFIG = low_rank_experiment_module.get_default_low_rank_config()
LOW_RANK_GRID_SEARCH = low_rank_experiment_module.get_default_low_rank_grid()
# Edit the value lists here. This is a 2x2x2x2 grid (16 trials).
LOW_RANK_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
LOW_RANK_RESULTS = low_rank_experiment_module.run_low_rank_grid_experiment(
    LOW_RANK_CONTEXT, base_config=LOW_RANK_CONFIG,
    search_space=LOW_RANK_GRID_SEARCH, display_fn=display
)


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.3878 val_loss=0.2177 val_MCC=0.7946 threshold=0.8360
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.2450 val_loss=0.1883 val_MCC=0.8025 threshold=0.6882
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.2068 val_loss=0.2034 val_MCC=0.8049 threshold=0.8510
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.1734 val_loss=0.2102 val_MCC=0.8058 threshold=0.8795
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.1383 val_loss=0.

"[LowRankFusion grid] trial_001/16 complete: {'rank': 8, 'fusion_dim': 256, 'dropout': 0.2, 'learning_rate': 0.0001}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.4729 val_loss=0.2336 val_MCC=0.7613 threshold=0.7384
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.2817 val_loss=0.2035 val_MCC=0.7934 threshold=0.7679
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.2443 val_loss=0.2095 val_MCC=0.8005 threshold=0.7799
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.2183 val_loss=0.2003 val_MCC=0.8056 threshold=0.7748
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.1950 val_loss=0.

"[LowRankFusion grid] trial_002/16 complete: {'rank': 8, 'fusion_dim': 256, 'dropout': 0.2, 'learning_rate': 5e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[LowRankFusion] A_no_geometry epoch 1/30 train_loss=0.3518 val_loss=0.2105 val_MCC=0.7953 threshold=0.8165
[LowRankFusion] A_no_geometry epoch 2/30 train_loss=0.2102 val_loss=0.1884 val_MCC=0.7986 threshold=0.7875
[LowRankFusion] A_no_geometry epoch 3/30 train_loss=0.1494 val_loss=0.2073 val_MCC=0.8014 threshold=0.7731
[LowRankFusion] A_no_geometry epoch 4/30 train_loss=0.0926 val_loss=0.2296 val_MCC=0.7946 threshold=0.8831
[LowRankFusion] A_no_geometry epoch 5/30 train_loss=0.0558 val_loss=0.

"[LowRankFusion grid] trial_003/16 complete: {'rank': 8, 'fusion_dim': 256, 'dropout': 0.1, 'learning_rate': 0.0001}"

... (25 outputs left out)

"[LowRankFusion grid] trial_016/16 complete: {'rank': 4, 'fusion_dim': 128, 'dropout': 0.1, 'learning_rate': 5e-05}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Source_File,Val_Accuracy,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Grid_Hyperparameters
0,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9407,0.8547,0.8301,0.9667,0.8422,0.9728,0.9107,trial_001,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
2,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9393,0.8325,0.8539,0.9595,0.8431,0.9718,0.9082,trial_002,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
8,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9408,0.8718,0.8087,0.9719,0.8391,0.9722,0.9094,trial_005,"{""dropout"": 0.2, ""fusion_dim"": 128, ""learning_..."
10,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9401,0.8593,0.8202,0.9683,0.8393,0.9721,0.9087,trial_006,"{""dropout"": 0.2, ""fusion_dim"": 128, ""learning_..."
16,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9394,0.8501,0.8286,0.9655,0.8392,0.9725,0.9071,trial_009,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
4,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9377,0.8248,0.8554,0.9572,0.8398,0.9709,0.9047,trial_003,"{""dropout"": 0.1, ""fusion_dim"": 256, ""learning_..."
18,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9363,0.8095,0.8711,0.9517,0.8391,0.9730,0.9074,trial_010,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
6,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9379,0.8405,0.8324,0.9627,0.8364,0.9703,0.9042,trial_004,"{""dropout"": 0.1, ""fusion_dim"": 256, ""learning_..."
24,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9376,0.8376,0.8347,0.9618,0.8362,0.9713,0.9045,trial_013,"{""dropout"": 0.2, ""fusion_dim"": 128, ""learning_..."
28,module06_architecture_run_20261102_012820,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,LowRankFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9378,0.8474,0.8221,0.9651,0.8346,0.9697,0.8996,trial_015,"{""dropout"": 0.1, ""fusion_dim"": 128, ""learning_..."


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,AUPRC,Trial_ID,Grid_Hyperparameters
97,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.9360,0.9633,0.9052,0.9662,0.9333,0.8735,0.9815,0.9812,trial_013,"{""dropout"": 0.2, ""fusion_dim"": 128, ""learning_..."
49,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.9289,0.9543,0.8994,0.9577,0.9260,0.8591,0.9839,0.9844,trial_007,"{""dropout"": 0.1, ""fusion_dim"": 128, ""learning_..."
73,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.9289,0.9408,0.9138,0.9437,0.9271,0.8580,0.9804,0.9812,trial_010,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
9,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.9275,0.9655,0.8851,0.9690,0.9235,0.8577,0.9826,0.9826,trial_002,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
1,module06_architecture_run_20261102_012820,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.9260,0.9684,0.8793,0.9718,0.9217,0.8555,0.9836,0.9837,trial_001,"{""dropout"": 0.2, ""fusion_dim"": 256, ""learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.7262,0.7362,0.7018,0.7504,0.7186,0.4527,0.7857,0.7622,trial_007,"{""dropout"": 0.1, ""fusion_dim"": 128, ""learning_..."
102,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.7254,0.7141,0.7485,0.7025,0.7309,0.4515,0.7921,0.7801,trial_013,"{""dropout"": 0.2, ""fusion_dim"": 128, ""learning_..."
30,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.7247,0.7281,0.7139,0.7354,0.7209,0.4494,0.7842,0.7639,trial_004,"{""dropout"": 0.1, ""fusion_dim"": 256, ""learning_..."
62,module06_architecture_run_20261102_012820,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,LowRankFusion,...,0.7232,0.7245,0.7169,0.7294,0.7207,0.4464,0.7848,0.7607,trial_008,"{""dropout"": 0.1, ""fusion_dim"": 128, ""learning_..."


'LowRankFusion grid complete (16 trials). Aggregate CSVs: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261102_012820\\module06_architecture_experiments\\low_rank'

## Ưu tiên đầu tiên tiếp theo — Hadamard fusion embedding

Chiếu DNA/protein delta riêng về cùng kích thước; tích từng chiều tạo tương tác DNA–protein. Skip path nối hai vector riêng để bảo toàn tín hiệu modality; bio/geometry vào classifier qua nhánh riêng. Grid 16 tổ hợp: hidden_dim {512, 256}, fusion_dim {512, 256}, dropout {0.2, 0.1}, learning_rate {1e-4, 5e-5}. Mỗi trial chạy A/B, chọn checkpoint theo validation MCC và đánh giá bốn tập test. Kết quả grid có CSV tổng hợp riêng, từng trial có checkpoint/manifest để resume.

In [8]:
import importlib
importlib.reload(importlib.import_module("core.module06_architecture_experiments.architecture_experiment_runner"))
hadamard_experiment_module = importlib.import_module("core.module06_architecture_experiments.hadamard_experiment")
hadamard_experiment_module = importlib.reload(hadamard_experiment_module)
HADAMARD_CONFIG = hadamard_experiment_module.get_default_hadamard_config()
HADAMARD_GRID_SEARCH = hadamard_experiment_module.get_default_hadamard_grid()
# Edit the candidate values here. This is a 2x2x2x2 grid (16 trials).
HADAMARD_CONTEXT = {
    'pipeline': pipeline, 'TARGET_CONFIGS': TARGET_CONFIGS, 'datasets': DATASETS,
    'DNA_MODEL': DNA_MODEL, 'PROTEIN_MODEL': PROTEIN_MODEL, 'DEVICE': DEVICE,
}
HADAMARD_RESULTS = hadamard_experiment_module.run_hadamard_grid_experiment(
    HADAMARD_CONTEXT, base_config=HADAMARD_CONFIG,
    search_space=HADAMARD_GRID_SEARCH, display_fn=display
)


[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.3245 val_loss=0.1952 val_MCC=0.7925 threshold=0.7430
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.2243 val_loss=0.2098 val_MCC=0.7904 threshold=0.8505
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.1879 val_loss=0.1875 val_MCC=0.7918 threshold=0.7729
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.1434 val_loss=0.2453 val_MCC=0.7841 threshold=0.9156
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.1026 val_lo

"[HadamardFusion grid] trial_001/16 complete: {'hidden_dim': 512, 'fusion_dim': 512, 'dropout': 0.2, 'learning_rate': 0.0001}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.3731 val_loss=0.2142 val_MCC=0.7699 threshold=0.7151
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.2479 val_loss=0.1814 val_MCC=0.7919 threshold=0.6732
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.2181 val_loss=0.1897 val_MCC=0.7946 threshold=0.7645
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.1900 val_loss=0.1971 val_MCC=0.7906 threshold=0.7026
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.1630 val_lo

"[HadamardFusion grid] trial_002/16 complete: {'hidden_dim': 512, 'fusion_dim': 512, 'dropout': 0.2, 'learning_rate': 5e-05}"

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['dna', 'prot', 'bio_core', 'spliceai']
  -> Đang nạp và tiền xử lý DNA Embeddings...
  -> Đang nạp và tiền xử lý Protein Embeddings...
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.

[HadamardFusion] A_no_geometry epoch 1/30 train_loss=0.3039 val_loss=0.1968 val_MCC=0.7974 threshold=0.7772
[HadamardFusion] A_no_geometry epoch 2/30 train_loss=0.2013 val_loss=0.2189 val_MCC=0.7871 threshold=0.8543
[HadamardFusion] A_no_geometry epoch 3/30 train_loss=0.1393 val_loss=0.1962 val_MCC=0.7791 threshold=0.6826
[HadamardFusion] A_no_geometry epoch 4/30 train_loss=0.0814 val_loss=0.2770 val_MCC=0.7701 threshold=0.9282
[HadamardFusion] A_no_geometry epoch 5/30 train_loss=0.0469 val_lo

"[HadamardFusion grid] trial_003/16 complete: {'hidden_dim': 512, 'fusion_dim': 512, 'dropout': 0.1, 'learning_rate': 0.0001}"

... (25 outputs left out)

"[HadamardFusion grid] trial_016/16 complete: {'hidden_dim': 256, 'fusion_dim': 256, 'dropout': 0.1, 'learning_rate': 5e-05}"

,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Pooling,DNA_Model,Prot_Model,Network,Architecture,...,Source_File,Val_Accuracy,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_AUROC,Val_AUPRC,Trial_ID,Grid_Hyperparameters
16,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9402,0.8644,0.8145,0.9699,0.8387,0.9715,0.9063,trial_009,"{""dropout"": 0.2, ""fusion_dim"": 512, ""hidden_di..."
12,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9391,0.8444,0.8344,0.9637,0.8393,0.9722,0.9089,trial_007,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
8,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9394,0.8554,0.8213,0.9673,0.8380,0.9724,0.9094,trial_005,"{""dropout"": 0.2, ""fusion_dim"": 256, ""hidden_di..."
22,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9385,0.8387,0.8393,0.9619,0.8390,0.9724,0.9065,trial_012,"{""dropout"": 0.1, ""fusion_dim"": 512, ""hidden_di..."
18,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9391,0.8507,0.8259,0.9658,0.8381,0.9723,0.9066,trial_010,"{""dropout"": 0.2, ""fusion_dim"": 512, ""hidden_di..."
20,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9391,0.8523,0.8236,0.9664,0.8377,0.9721,0.9069,trial_011,"{""dropout"": 0.1, ""fusion_dim"": 512, ""hidden_di..."
14,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9383,0.8371,0.8397,0.9615,0.8384,0.9728,0.9084,trial_008,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
10,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9388,0.8490,0.8259,0.9654,0.8373,0.9720,0.9063,trial_006,"{""dropout"": 0.2, ""fusion_dim"": 256, ""hidden_di..."
30,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9371,0.8266,0.8481,0.9581,0.8372,0.9727,0.9090,trial_016,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
28,module06_architecture_run_20261102_022601,DL,validation,A_no_geometry,bio_core_dna_prot_spliceai,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,HadamardFusion,...,D:\Missense_variant_pathogenicity_prediction\e...,0.9374,0.8326,0.8412,0.9601,0.8369,0.9718,0.9069,trial_015,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."


,Run_ID,Branch,Evaluation_Split,Target_Config,Ablation,Dataset,Pooling,DNA_Model,Prot_Model,Network,...,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,AUPRC,Trial_ID,Grid_Hyperparameters
89,module06_architecture_run_20261102_022601,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.9289,0.9627,0.8908,0.9662,0.9254,0.8600,0.9830,0.9828,trial_012,"{""dropout"": 0.1, ""fusion_dim"": 512, ""hidden_di..."
121,module06_architecture_run_20261102_022601,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.9289,0.9515,0.9023,0.9549,0.9263,0.8588,0.9816,0.9819,trial_016,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
113,module06_architecture_run_20261102_022601,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.9260,0.9540,0.8937,0.9577,0.9228,0.8536,0.9804,0.9803,trial_015,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
97,module06_architecture_run_20261102_022601,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.9246,0.9511,0.8937,0.9549,0.9215,0.8506,0.9803,0.9798,trial_013,"{""dropout"": 0.2, ""fusion_dim"": 256, ""hidden_di..."
57,module06_architecture_run_20261102_022601,DL,test,A_no_geometry,bio_core_dna_prot_spliceai,Train3Val_ClinVarHQ,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.9203,0.9563,0.8793,0.9606,0.9162,0.8432,0.9797,0.9790,trial_008,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46,module06_architecture_run_20261102_022601,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.7292,0.7244,0.7364,0.7220,0.7304,0.4585,0.7959,0.7868,trial_006,"{""dropout"": 0.2, ""fusion_dim"": 256, ""hidden_di..."
54,module06_architecture_run_20261102_022601,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.7284,0.7352,0.7108,0.7459,0.7228,0.4570,0.7912,0.7834,trial_007,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."
86,module06_architecture_run_20261102_022601,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.7277,0.7378,0.7033,0.7519,0.7201,0.4558,0.7891,0.7731,trial_011,"{""dropout"": 0.1, ""fusion_dim"": 512, ""hidden_di..."
62,module06_architecture_run_20261102_022601,DL,test,B_with_geometry,bio_core_dna_prot_spliceai_geom,Train3Val_UniProt,center,nt_v1_500m_dna_1001,esm2_650m_prot_501,HadamardFusion,...,0.7232,0.7179,0.7319,0.7145,0.7248,0.4465,0.7909,0.7826,trial_008,"{""dropout"": 0.1, ""fusion_dim"": 256, ""hidden_di..."


'HadamardFusion grid complete (16 trials). Aggregate CSVs: D:\\Missense_variant_pathogenicity_prediction\\experiments\\module06_architecture_run_20261102_022601\\module06_architecture_experiments\\hadamard'